# Day 7: Incremental Processing & CDC Upsert Engine

This notebook demonstrates idempotent UPSERT/DELETE logic on a simulated stateful table using Iceberg's MERGE INTO.

In [ ]:
import os
from pyspark.sql import SparkSession

warehouse_path = "/tmp/iceberg_warehouse"
os.makedirs(warehouse_path, exist_ok=True)

spark = SparkSession.builder \
    .appName("Day07_Incremental_Merge") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Setup Target Silver Table (Initial State)
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.silver")
spark.sql("DROP TABLE IF EXISTS local.silver.person")

spark.sql("""
CREATE TABLE local.silver.person (
    person_id STRING,
    name STRING,
    updated_at TIMESTAMP
)
USING iceberg
""")

spark.sql("""
INSERT INTO local.silver.person VALUES 
    ('1', 'Alice Old', timestamp '2023-01-01 10:00:00'),
    ('2', 'Bob', timestamp '2023-01-01 10:00:00')
""")

print("INITIAL TARGET TABLE STATE:")
spark.sql("SELECT * FROM local.silver.person ORDER BY person_id").show()

In [ ]:
# Simulate Incoming Deduplicated CDC Batch
# 'op' indicates operation: 'c' (create), 'u' (update), 'd' (delete)
spark.sql("""
CREATE OR REPLACE TEMP VIEW new_cdc_events AS
SELECT * FROM VALUES 
    ('1', 'Alice New', timestamp '2023-01-02 10:00:00', 'u'), -- Update (valid, newer timestamp)
    ('2', 'Bob Delete', timestamp '2023-01-02 11:00:00', 'd'), -- Delete
    ('3', 'Charlie', timestamp '2023-01-02 12:00:00', 'c'),    -- Create new
    ('1', 'Alice Ignored', timestamp '2022-12-31 10:00:00', 'u') -- Update (ignored, out-of-order old timestamp)
AS tab(person_id, name, updated_at, op)
""")

print("INCOMING CDC BATCH:")
spark.sql("SELECT * FROM new_cdc_events").show()

In [ ]:
# Perform MERGE INTO
merge_sql = """
MERGE INTO local.silver.person t
USING new_cdc_events s
ON t.person_id = s.person_id
WHEN MATCHED AND s.op = 'd' AND s.updated_at > t.updated_at THEN
    DELETE
WHEN MATCHED AND s.op IN ('u', 'c') AND s.updated_at > t.updated_at THEN
    UPDATE SET t.name = s.name, t.updated_at = s.updated_at
WHEN NOT MATCHED AND s.op IN ('c', 'u') THEN
    INSERT (person_id, name, updated_at) VALUES (s.person_id, s.name, s.updated_at)
"""
spark.sql(merge_sql)

print("TARGET TABLE STATE AFTER MERGE:")
spark.sql("SELECT * FROM local.silver.person ORDER BY person_id").show()